#### 14. **parity** (Number of previous births)
- Total live births given by a woman before the current one.

- **NDHS 2022:** Average number of children per woman ($TFR$): ~2.1 (lower in urban/richest, higher in rural/poorest; ranges 0-7 for simulation).

- **Distribution:** Skewed right. Poisson (λ ≈ 2.3).

#### 15. **anc_visits** (Antenatal care visits)
- **NDHS 2022:** 81% have at least one ANC; **69% have ≥4 visits** (urban/richest more, rural/poorest less).

- **Distribution:** ~12% have 0–3 visits, ~19% have 4–7, ~69% have 8+.

- Typically capped at 12 for simulation.

#### 16. **delivery_type** 
- **NDHS 2022:** **Normal: 70%, Cesarean: 25%, Assisted: 5%** overall; CS much higher in private hospitals (up to 50%). Cesarean more likely for higher wealth/urban.

- **Distribution:** National: 70/25/5.

#### 17. **birth_complications**  
- **Prevalence:** ~25% of all deliveries report complications (higher among Cesareans; up to 45% with CS, ~19% with normal/assisted).

- **Distribution:** Yes: 25%, mapped higher if cesarean.

#### 18. **baby_health_status**
- **NDHS/WHO Nepal:** **Healthy: 80%, Minor issues: 15%, Major issues: 5%.**
- Major issues include NICU admission, respiratory distress, etc..

#### 19. **gestational_age**
- **Normal (NDHS, clinical studies):** Median: 39 weeks, SD ~2. Normal range: **28–42 weeks**. Preterm birth 10%, postterm rare.

#### 20. **maternal_age_at_first_birth**
- **NDHS 2022:** **Median: 20.6 years**. Urban: 21.1, Rural: 19.9. Early motherhood <20; late >30 uncommon.

#### 21. **previous_pregnancy_loss**
- **NDHS 2022:** **20%** of women report a previous pregnancy loss (miscarriage, stillbirth, abortion) in last 3 years.

### **Relation Mapping Based on NDHS and Research**

| Variable                     | Strongly Related to...                                                               | Observed Nepal Patterns                                                        |
|------------------------------|--------------------------------------------------------------------------------------|-------------------------------------------------------------------------------|
| **parity**                   | maternal_age, wealth, residence, education                                           | Higher age/rural/poorer = higher parity. Urban = fewer children.               |
| **anc_visits**               | wealth, residence, education, age, parity                                            | Urban/rich/educated → more visits. Rural/poor/parity>2 → fewer visits.         |
| **delivery_type**            | wealth, residence, age, parity, anc_visits                                           | CS higher in urban, private, high-wealth; low ANC/poor = fewer CS, more normal.|
| **birth_complications**      | delivery_type, parity, age                                                           | More likely with CS, advanced age, higher parity.                              |
| **baby_health_status**       | birth_complications, delivery type, parity                                           | Complications ↑ = worse baby health. CS/assisted ↑ = minor/major issues.       |
| **gestational_age**          | complications, parity, age                                                           | Preterm birth higher with complications, parity >3.                            |
| **maternal_age_at_first_birth** | parity, education, region                                                         | Lower in rural/poor/less educated; ties to higher total parity.                |
| **previous_pregnancy_loss**  | parity, complications, maternal age, residence                                      | More likely with higher parity/age/complications; rural/poor slightly ↑ risk.  |

#### **Summary Table of Typical Ranges**

| Feature                  | Typical Nepal Range / Value |
|--------------------------|----------------------------|
| parity                   | 0–7 (mean ~2.1)            |
| anc_visits               | 0–12 (median 5, ~70% ≥4)   |
| delivery_type            | Normal 70%, CS 25%, Assisted 5% |
| birth_complications      | Yes: 25%, (↑ for CS)       |
| baby_health_status       | Healthy 80%, Minor 15%, Major 5% |
| gestational_age (weeks)  | 28–42 (mean ~39, sd~2)     |
| maternal_age_at_first_birth | 15–32 (median 20.6)    |
| previous_pregnancy_loss  | Yes: 20%                   |

---
----
-----

In [1]:
import numpy as np
import pandas as pd

n_samples = 15000

In [3]:
def realistic_parity(age, wealth_index):
    # Older and poorer → higher parity; younger/urban/rich → less
    lam = 2.8 if age > 30 else 2.1
    if wealth_index in ("Richest", "Richer"):
        lam -= 0.4
    if wealth_index in ("Poorest", "Poorer"):
        lam += 0.3
    return np.clip(np.random.poisson(lam), 0, 7)

def anc_visits(wealth_index, residence, parity):
    # NDHS: 69% >=4, higher in rich/urban/low-parity
    if wealth_index in ("Richest", "Richer") and residence == "Urban" and parity < 3:
        mean, sd = 7.0, 2.0
    elif wealth_index == "Poorest" or residence == "Rural":
        mean, sd = 4.2, 2.1
    else:
        mean, sd = 5.5, 2.1
    val = int(np.round(np.clip(np.random.normal(mean, sd), 0, 12)))
    return val

def delivery_type(wealth_index, anc_visits, age):
    # NDHS: Normal 70%, CS 25%, Assisted 5%; CS↑ for rich, more ANC, age↑
    cs_rate = 0.31 if wealth_index == "Richest" else 0.22
    assisted = 0.08 if anc_visits < 2 else 0.04
    if age > 34: cs_rate += 0.06
    r = np.random.rand()
    if r < cs_rate: return "Cesarean"
    elif r < cs_rate + assisted: return "Assisted"
    else: return "Normal"

def birth_complications(delivery_type, parity):
    # 45% with CS, 19% with normal/assisted; parity>3 ↑ risk
    risk = 0.45 if delivery_type == "Cesarean" else 0.19
    if parity > 3: risk += 0.07
    return np.random.rand() < risk

def baby_health_status(birth_complications, delivery_type):
    # Complications ↑ minor/major issues; CS ↑ minor, Assisted ↑ major
    if not birth_complications:
        return np.random.choice(["Healthy", "Minor issues", "Major issues"], p=[0.88, 0.09, 0.03])
    else:
        if delivery_type == "Assisted":
            return np.random.choice(["Healthy", "Minor issues", "Major issues"], p=[0.41, 0.37, 0.22])
        elif delivery_type == "Cesarean":
            return np.random.choice(["Healthy", "Minor issues", "Major issues"], p=[0.49, 0.41, 0.10])
        else:
            return np.random.choice(["Healthy", "Minor issues", "Major issues"], p=[0.60, 0.29, 0.11])

def gestational_age(baby_health_status):
    # More issues/major issues + preterm (<37w) more likely
    base = np.random.normal(39, 2)
    if baby_health_status == "Healthy":
        return int(np.clip(base, 38, 42))
    elif baby_health_status == "Minor issues":
        return int(np.clip(base - np.random.rand()*1.5, 36, 40))
    else:
        return int(np.clip(base - 2.2 + np.random.rand()*1.2, 28, 38))

def maternal_age_at_first_birth(age, parity):
    # Median is 20.6, high parity → younger AFB, older age → AFB earlier
    if parity == 0:
        return np.random.randint(15, age)
    mean_afb = 20.5 - 0.07 * parity + 0.16 * (age-28)
    afb = int(np.clip(np.random.normal(mean_afb, 2.1), 15, age-2))
    return afb

def previous_pregnancy_loss(parity, birth_complications):
    # 20% overall, higher for parity>2 or recent complications
    p = 0.16 + 0.07*(parity>2) + 0.06*birth_complications
    return np.random.rand() < p

# Simulate realistic "residence", "wealth", and "age" for dependencies
wealth_cats = ['Poorest','Poorer','Middle','Richer','Richest']
wealth_p = [0.202,0.202,0.202,0.197,0.197]
residences = ["Urban","Rural"]
resid_p = [0.65, 0.35]

rows = []
for i in range(n_samples):
    age = np.random.randint(17, 43) # Nepal avg childbearing age ~26
    wealth = np.random.choice(wealth_cats, p=wealth_p)
    residence = np.random.choice(residences, p=resid_p)
    parity = realistic_parity(age, wealth)
    anc = anc_visits(wealth, residence, parity)
    d_type = delivery_type(wealth, anc, age)
    comp = birth_complications(d_type, parity)
    baby_stat = baby_health_status(comp, d_type)
    gestational = gestational_age(baby_stat)
    afb = maternal_age_at_first_birth(age, parity)
    p_loss = previous_pregnancy_loss(parity, comp)
    rows.append([
        parity, anc, d_type, comp, baby_stat, gestational, afb, p_loss, age
    ])

columns = [
    'parity',
    'anc_visits',
    'delivery_type',
    'birth_complications',
    'baby_health_status',
    'gestational_age',
    'maternal_age_at_first_birth',
    'previous_pregnancy_loss',
    'mother_age'
]

df = pd.DataFrame(rows, columns=columns)

# Clean up formats for binary columns
df['birth_complications'] = df['birth_complications'].map({True: "Yes", False: "No"})
df['previous_pregnancy_loss'] = df['previous_pregnancy_loss'].map({True: "Yes", False: "No"})

In [4]:
df.head()

,parity,anc_visits,delivery_type,birth_complications,baby_health_status,gestational_age,maternal_age_at_first_birth,previous_pregnancy_loss,mother_age
0,1,3,Normal,Yes,Minor issues,37,15,No,19
1,0,2,Cesarean,No,Major issues,37,17,No,38
2,1,6,Normal,No,Healthy,38,16,No,18
3,1,9,Normal,Yes,Healthy,39,18,No,37
4,2,6,Cesarean,Yes,Healthy,38,20,No,27


- **Mirrors Nepal’s NDHS 2022 distributions** for childbearing, complications, baby status, cesarean rates, ANC visits, etc.

- **Imposes dependencies**: e.g., poor and older = higher parity, more complications with CS, parity or urban/rich, issues linked logically across birth features.

- **Easy to integrate**: Can be joined with other demographic/economic columns by age/row.